In [3]:
import os

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from torch.nn.functional import softmax
from torch.optim import AdamW
from torch.utils.data import DataLoader

from internal.data_types import HistologyDataset
from internal.nn.model import train_one_epoch, validate
from internal.persistence_manager import PersistenceManager
from notebooks.internal.nn.weighted_random_sampler import make_weighted_sampler

data = PersistenceManager.load_dataset()
test_df = data.test_df
train_df = data.train_df
train_transforms = data.train_transforms
val_test_transforms = data.val_test_transforms
idx2label = data.idx2label

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cuda_is_available = torch.cuda.is_available()
print(f'Using device: {device}')

Arrays and scalers loaded successfully from: /home/andre/university/AN2DL-Challenge-2/notebooks/processed/dataset.joblib
Using device: cuda


In [4]:
N_FOLDS = data.num_K_folds
IMAGE_SIZE = data.image_size
BATCH_SIZE = 4
PRETRAINED_MODEL = "tf_efficientnetv2_s.in21k"
N_CLASSES = 4 # number of classes in the dataset (labels)
N_WORKERS = os.cpu_count() // 2 if os.cpu_count() else 4

for fold in range(N_FOLDS):
    print(f"\n========== Fold {fold} ==========")

    train_df_split = train_df[train_df["fold"] != fold].reset_index(drop=True)
    val_df_split   = train_df[train_df["fold"] == fold].reset_index(drop=True)

    train_dataset = HistologyDataset(train_df_split, transforms=train_transforms, is_train=True, image_size=IMAGE_SIZE)
    val_dataset   = HistologyDataset(val_df_split,   transforms=val_test_transforms, is_train=True, image_size=IMAGE_SIZE)

    sampler = make_weighted_sampler(train_df_split)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        num_workers=N_WORKERS,
        pin_memory=cuda_is_available
    )
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=N_WORKERS, pin_memory=cuda_is_available)

    # --- create fresh model for this fold ---
    model = timm.create_model(
        PRETRAINED_MODEL,
        pretrained=True,
        num_classes=N_CLASSES
    ).to(device)

    # --- Stage 1: freeze backbone, train classifier head ---
    print("\n--- Stage 1: Training classifier head ---")

    # --- 1.1. freeze feature extractor layers ---
    for param in model.parameters():
        param.requires_grad = False

    # 2) unfreeze classifier head (EffNetV2 uses .classifier)
    for param in model.classifier.parameters():
        param.requires_grad = True

    # --- 1.2. define loss, optimizer, scheduler ---
    criterion = nn.CrossEntropyLoss()
    head_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(head_params, lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=10
    )

    # --- 1.3. train for several epochs ---
    EPOCHS = 8
    best_f1 = 0.0
    best_state = None
    for epoch in range(1, EPOCHS+1):
        print(f"\nEpoch {epoch}/{EPOCHS}")
        train_loss, train_acc, train_f1 = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )
        val_loss, val_acc, val_f1 = validate(
            model, val_loader, criterion, device
        )
        scheduler.step()
        print(
            f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
            f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
        )
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = model.state_dict().copy()
            torch.save(best_state, f"best_effv2_stage1_fold{fold}_f1_{val_f1:.4f}.pth")
            print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

    # --- Stage 2: unfreeze whole model, fine-tune ---
    print("\n--- Stage 2: Fine-tuning entire model ---")

    # --- 2.1. unfreeze entire model ---
    for param in model.parameters():
        param.requires_grad = True

    # --- 2.2. define loss, optimizer, scheduler ---
    optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=15
    )

    # --- 2.3. mild class weights ---
    class_counts = torch.tensor([445, 414, 397, 156], dtype=torch.float32) # 445 LumB, 414 LumA, 397 Her2, 156 TN
    class_weights = (class_counts.sum() / class_counts)
    class_weights = class_weights / class_weights.mean()
    # criterion = FocalLoss(alpha=class_weights, gamma=2.0)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.1)

    # --- 2.4. train for several epochs ---
    EPOCHS = 15
    best_f1 = 0.0
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        print(f"\nEpoch {epoch}/{EPOCHS}")
        train_loss, train_acc, train_f1 = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )
        val_loss, val_acc, val_f1 = validate(
            model, val_loader, criterion, device
        )
        scheduler.step()
        print(
            f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
            f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
        )
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = model.state_dict().copy()
            torch.save(best_state, f"best_effv2_stage2_fold{fold}_f1_{val_f1:.4f}.pth")
            print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

    # --- save model for this fold ---
    torch.save(model.state_dict(), f"effv2_s_fold{fold}.pth")


========== Fold 0 ==========

--- Stage 1: Training classifier head ---

Epoch 1/8


    t_loss=4.5071 | F1(macro)=0.2355 | Acc=0.2356


Confusion matrix:
 [[28  4  8 49]
 [28 11  4 40]
 [22 11  6 40]
 [ 8  1  2 21]]
Train  loss=4.5071 acc=0.2356 f1=0.2355 | Val loss=6.7114 acc=0.2332 f1=0.2180
  🔥 New best F1: 0.2180 – model saved.

Epoch 2/8


    t_loss=3.2626 | F1(macro)=0.3056 | Acc=0.3065


Confusion matrix:
 [[17  5  8 59]
 [21 18  7 37]
 [16 10  9 44]
 [ 3  1  2 26]]
Train  loss=3.2626 acc=0.3065 f1=0.3056 | Val loss=6.5027 acc=0.2473 f1=0.2437
  🔥 New best F1: 0.2437 – model saved.

Epoch 3/8


...

Confusion matrix:
 [[ 8 17 15 49]
 [12 31  9 31]
 [ 9 15 23 32]
 [ 5  1  5 20]]
Train  loss=0.9920 acc=0.5823 f1=0.5655 | Val loss=1.7119 acc=0.2908 f1=0.2871
  🔥 New best F1: 0.2871 – model saved.

Epoch 14/15


    t_loss=0.9993 | F1(macro)=0.5637 | Acc=0.5743


Confusion matrix:
 [[10 14 10 55]
 [15 24 15 29]
 [12 14 26 27]
 [ 4  0  6 21]]
Train  loss=0.9993 acc=0.5743 f1=0.5637 | Val loss=1.7943 acc=0.2872 f1=0.2874
  🔥 New best F1: 0.2874 – model saved.

Epoch 15/15


    t_loss=0.9927 | F1(macro)=0.5565 | Acc=0.5664


Confusion matrix:
 [[11 10 16 52]
 [14 26 13 30]
 [13  8 19 39]
 [ 7  1  4 19]]
Train  loss=0.9927 acc=0.5664 f1=0.5565 | Val loss=1.8046 acc=0.2660 f1=0.2707


In [5]:
test_dataset = HistologyDataset(
    test_df,
    transforms=val_test_transforms,
    is_train=False,   # returns (img, sample_index)
    image_size=IMAGE_SIZE
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=N_WORKERS,
    pin_memory=cuda_is_available
)

all_fold_probs = []   # list of arrays [N, num_classes]
all_sample_indices = None

for fold in range(N_FOLDS):
    print(f"Inference with fold {fold} model")

    # recreate model and load weights
    model = timm.create_model(
        PRETRAINED_MODEL,
        pretrained=False,
        num_classes=N_CLASSES
    ).to(device)
    state = torch.load(f"effv2_s_fold{fold}.pth", map_location=device)
    model.load_state_dict(state)
    model.eval()

    fold_probs = []
    sample_indices_list = []

    with torch.no_grad():
        for imgs, sample_indices in test_loader:
            imgs = imgs.to(device, non_blocking=True)

            logits = model(imgs)               # [B, num_classes]
            probs = softmax(logits, dim=1)     # [B, num_classes]
            fold_probs.append(probs.cpu().numpy())

            # collect sample indices only once
            if all_sample_indices is None:
                sample_indices_list.extend(sample_indices)

    fold_probs = np.concatenate(fold_probs, axis=0)  # [N, num_classes]
    all_fold_probs.append(fold_probs)

    if all_sample_indices is None:
        all_sample_indices = sample_indices_list

# average probabilities across folds
mean_probs = np.mean(all_fold_probs, axis=0)   # [N, num_classes]
pred_indices = mean_probs.argmax(axis=1)

pred_labels = [idx2label[int(i)] for i in pred_indices]
sample_index_with_ext = [
    f"{si}.png" if not si.endswith(".png") else si
    for si in all_sample_indices
]

submission_df = pd.DataFrame({
    "sample_index": sample_index_with_ext,
    "label": pred_labels
})

submission_df.to_csv("submission_5fold_no_tta.csv", index=False)
print("Saved submission_5fold_no_tta.csv")
print(submission_df.head())


Inference with fold 0 model
Inference with fold 1 model
Inference with fold 2 model
Inference with fold 3 model
Inference with fold 4 model
Saved submission_5fold_no_tta.csv
   sample_index            label
0  img_0000.png        Luminal B
1  img_0001.png  Triple negative
2  img_0002.png  Triple negative
3  img_0003.png  Triple negative
4  img_0004.png        Luminal B


In [6]:
########################################################
# ===== Inference with TTA and 5-Fold Ensembling ===== #
########################################################
# all_fold_probs = []
# all_sample_indices = None
#
# test_dataset = HistologyDataset(test_df, transforms=val_test_transforms, is_train=False, image_size=IMAGE_SIZE)
# test_loader = DataLoader(test_dataset, batch_size=1,  # IMPORTANT: batch_size=1 for per-image TTA
#                          shuffle=False, num_workers=N_WORKERS, pin_memory=cuda_is_available)
#
# for fold in range(N_FOLDS):
#     print(f"Inference with fold {fold} model")
#     model = timm.create_model(PRETRAINED_MODEL, pretrained=False, num_classes=N_CLASSES).to(device)
#     model.load_state_dict(torch.load(f"effv2_s_fold{fold}.pth", map_location=device))
#     model.eval()
#
#     fold_probs = []
#     sample_indices_list = []
#
#     with torch.no_grad():
#         for img_tensor, sample_idx in test_loader:
#             img_tensor = img_tensor.squeeze(0)  # [3,H,W]
#             img_tensor = img_tensor.to(device)
#
#             # -------- TTA: apply multiple augmented views --------
#             tta_tensors = apply_tta(img_tensor)
#
#             # accumulate probability predictions
#             probs_sum = 0
#             for aug_img in tta_tensors:
#                 aug_img = aug_img.unsqueeze(0).to(device)  # [1,3,H,W]
#                 logits = model(aug_img)
#                 probs = softmax(logits, dim=1)  # [1,4]
#                 probs_sum += probs[0].cpu().numpy()
#
#             # average across TTA views
#             avg_probs = probs_sum / len(tta_tensors)
#             fold_probs.append(avg_probs)
#
#             if all_sample_indices is None:
#                 sample_indices_list.append(sample_idx[0])
#
#     fold_probs = np.vstack(fold_probs)  # [N, 4]
#     all_fold_probs.append(fold_probs)
#
#     if all_sample_indices is None:
#         all_sample_indices = sample_indices_list
#
# mean_probs = np.mean(all_fold_probs, axis=0)  # [N, 4]
# pred_indices = mean_probs.argmax(axis=1)
# pred_labels = [idx2label[int(i)] for i in pred_indices]
#
# sample_index_with_ext = [
#     f"{si}.png" if not si.endswith(".png") else si
#     for si in all_sample_indices
# ]
#
# submission_df = pd.DataFrame({
#     "sample_index": sample_index_with_ext,
#     "label": pred_labels
# })
# submission_df.to_csv("submission_5fold_tta.csv", index=False)
#
# print("Saved submission_5fold_tta.csv")